# Step 0 — Get a boundary GeoJSON

Every step in this pipeline starts from a **boundary GeoJSON file** — a polygon that
defines your area of interest. This notebook shows four ways to create one.

| Method | Best for | Requires |
|---|---|---|
| **Method A** — Named place (OSMnx) | Any named admin area directly from OSM | `pip install osmnx` |
| **Method B** — Bounding box | Custom rectangular region from coordinates | nothing extra |
| **Method C** — Local admin file | Extracting from GADM / Geofabrik / national data | a GeoJSON file |
| **Method D** — osm-boundaries.com | Browse + download all admin levels for any country | download from website |

Run whichever method fits your use case, then run the **Inspect** and **Save** cells at the bottom.

In [1]:
%%time
from pathlib import Path
import geopandas as gpd
import folium

# ── Configuration ─────────────────────────────────────────────────────────
NAME         = 'sodermalm'          # used for the output filename
SEARCH_NAME         = 'Södermalm'   # change to match a name in the table printed below

BOUNDARY_DIR = Path('../boundaries')
# ─────────────────────────────────────────────────────────────────────────

BOUNDARY_DIR.mkdir(exist_ok=True)
OUTPUT_PATH  = BOUNDARY_DIR / f'{NAME}.geojson'
boundary_gdf = None   # set by whichever Method cell you run below

print(f'Output will be saved to: {OUTPUT_PATH}')

Output will be saved to: ../boundaries/sodermalm.geojson
CPU times: user 2 s, sys: 107 ms, total: 2.11 s
Wall time: 578 ms


---
## Method A — Named place via OpenStreetMap

**Easiest method.** `osmnx` queries the [Nominatim](https://nominatim.openstreetmap.org/)
geocoding service and fetches the official administrative boundary for any named place
directly from OpenStreetMap.

Works for: cities, districts, municipalities, counties, countries.

```
pip install osmnx
```

**Tip:** If your query returns multiple results (e.g. a city name shared by several countries),
add more context: `'Södermalm, Stockholm, Sweden'`.

In [ ]:
%%time
import osmnx as ox

# Change this to your area of interest
PLACE_NAME = 'Södermalm, Stockholm, Sweden'

boundary_a = ox.geocode_to_gdf(PLACE_NAME)
print(f'Found: {PLACE_NAME}')
print(f'CRS: {boundary_a.crs}')
print(f'Bounds: {dict(zip(["W","S","E","N"], boundary_a.total_bounds.round(5)))}')
boundary_a.plot(figsize=(7, 5), color='steelblue', alpha=0.4, edgecolor='navy')
import matplotlib.pyplot as plt
plt.title(PLACE_NAME); plt.show()

# Assign to the shared variable used by the Save cell below
boundary_gdf = boundary_a[['geometry']].to_crs('EPSG:4326')

---
## Method B — Bounding box

Define your area as a rectangle using the four corner coordinates.
Useful when you want a precise rectangular region, or when Method A
returns a boundary that is too large or oddly shaped.

**How to find coordinates:**
- Open [OpenStreetMap](https://www.openstreetmap.org/) or Google Maps
- Right-click on the map → copy coordinates (lat, lon)
- Or use [bboxfinder.com](http://bboxfinder.com/) to draw and copy a box

In [ ]:
%%time
from shapely.geometry import box

# Coordinates in WGS-84 (EPSG:4326): west, south, east, north
WEST  = 18.0248
SOUTH = 59.3027
EAST  = 18.1093
NORTH = 59.3238

bbox_geom   = box(WEST, SOUTH, EAST, NORTH)
boundary_b  = gpd.GeoDataFrame(geometry=[bbox_geom], crs='EPSG:4326')

center = [(SOUTH + NORTH) / 2, (WEST + EAST) / 2]
m = folium.Map(location=center, zoom_start=12, tiles='OpenStreetMap')
folium.GeoJson(boundary_b.__geo_interface__,
               style_function=lambda _: {'color':'navy','weight':2,'fillOpacity':0.15},
               tooltip='Bounding box').add_to(m)
display(m)

# Assign to the shared variable
boundary_gdf = boundary_b

---
## Method C — Extract from an admin boundaries file

If you have a GeoJSON file with multiple administrative polygons
(e.g. all municipalities in a country), you can search by name and extract one.

**Where to get admin boundary files:**
- [GADM](https://gadm.org/download_country.html) — administrative boundaries for every country
- [Geofabrik](https://download.geofabrik.de/) — OSM extracts including boundary relations
- [Natural Earth](https://www.naturalearthdata.com/) — country/region polygons
- National open data portals (e.g. Statistics Sweden for Swedish municipalities)

In [ ]:
%%time
# Path to a GeoJSON with multiple polygons (e.g. all Swedish municipalities)
ADMIN_FILE  = Path('../data/sweden-admin7.geojson')   # change this
NAME_COLUMN = 'name'                                   # column that holds the name

if not ADMIN_FILE.exists():
    print(f'File not found: {ADMIN_FILE}')
    print('Download from GADM (https://gadm.org) or your national data portal.')
else:
    admin = gpd.read_file(ADMIN_FILE).to_crs('EPSG:4326')
    print(f'Loaded {len(admin)} polygons. Columns: {list(admin.columns)}')

    # Search — case-insensitive partial match
    matches = admin[admin[NAME_COLUMN].str.contains(SEARCH_NAME, case=False, na=False)]
    print(f'Found {len(matches)} match(es):')
    print(matches[[NAME_COLUMN]].to_string())

    # Pick the first match (or adjust index if multiple results)
    boundary_c  = matches.iloc[[0]][['geometry']]
    boundary_gdf = boundary_c

    boundary_gdf.plot(figsize=(7, 5), color='steelblue', alpha=0.4, edgecolor='navy')
    plt.title(SEARCH_NAME); plt.show()

---
## Method D — osm-boundaries.com (local file)

**[osm-boundaries.com](https://osm-boundaries.com/map)** lets you download all OSM
administrative boundaries for a country at a given admin level as a single file.

### Steps to download

1. Go to **https://osm-boundaries.com/map**
2. Click **Select country** → search for your country (e.g. *Sweden*)
3. Choose the **admin level**:

| Admin level | Typical meaning (varies by country) |
|---|---|
| 4 | Country |
| 6 | Region / county |
| 7 | Municipality |
| 8 | District / borough / suburb |
| 9–10 | Neighbourhood |

4. Click **Download** → save to the `boundaries/` folder of this project

The file can be `.geojson` or `.gz` (gzip-compressed GeoJSON) — geopandas reads both directly.

### Then run the cell below

It loads the file, lists all available area names so you can browse them,
then extracts the one you need.

In [ ]:
%%time
import gzip, json

# Admin level 7 = municipalities (e.g. Stockholm, Nacka)
# Admin level 8 = districts      (e.g. Södermalm, Kungsholmen)
OSM_BOUNDARIES_FILE = Path('../boundaries/sweden-admin8.gz')
SEARCH_NAME         = 'Södermalm'   # names starting with this string will be matched

boundary_gdf = None

if not OSM_BOUNDARIES_FILE.exists():
    print(f'File not found: {OSM_BOUNDARIES_FILE}')
    print('Go to https://osm-boundaries.com/map → select country + admin level → Download → save to boundaries/')
else:
    open_fn = gzip.open if OSM_BOUNDARIES_FILE.suffix == '.gz' else open
    with open_fn(OSM_BOUNDARIES_FILE, 'rt', encoding='utf-8') as f:
        geojson = json.load(f)

    admin = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
    print(f'Loaded {len(admin)} boundaries from {OSM_BOUNDARIES_FILE.name}')

    name_cols = [c for c in ['name', 'name_en', 'osm_id'] if c in admin.columns]
    print('\nAll available areas:')
    display(admin[name_cols].sort_values('name').reset_index(drop=True))

    # ── startswith match (case-insensitive) ───────────────────────────────
    mask = admin['name'].str.startswith(SEARCH_NAME, na=False)
    if not mask.any() and 'name_en' in admin.columns:
        mask = admin['name_en'].str.startswith(SEARCH_NAME, na=False)

    matches = admin[mask]
    print(f'\nMatches starting with "{SEARCH_NAME}": {len(matches)} found')

    if len(matches) == 0:
        print('No match — check the table above for the correct name spelling.')
    else:
        display(matches[name_cols].reset_index(drop=True))
        boundary_gdf = matches.iloc[[0]][['geometry']]
        import matplotlib.pyplot as plt
        boundary_gdf.plot(figsize=(7, 5), color='steelblue', alpha=0.4, edgecolor='navy')
        plt.title(matches.iloc[0]['name']); plt.show()

---
## Inspect the boundary

Before saving, verify the boundary looks correct on an interactive map.
Check that:
- The polygon covers your intended area (not too large, not too small)
- It has no obvious gaps or extra fragments
- The bounding box coordinates look reasonable

In [3]:
%%time
if boundary_gdf is None:
    print('boundary_gdf is not set — run one of the Method cells above first.')
    print('If using Method D, check that SEARCH_NAME matches a name in the printed table.')
else:
    west, south, east, north = boundary_gdf.total_bounds
    center = [(south + north) / 2, (west + east) / 2]

    print('Boundary bounds:')
    print(f'  West:  {west:.5f}°   East:  {east:.5f}°')
    print(f'  South: {south:.5f}°   North: {north:.5f}°')
    print(f'  Width: {(east-west)*111:.1f} km   Height: {(north-south)*111:.1f} km  (approx)')

    m = folium.Map(location=center, zoom_start=12, tiles='OpenStreetMap')
    folium.GeoJson(
        boundary_gdf.__geo_interface__,
        style_function=lambda _: {'color': 'navy', 'weight': 2, 'fillOpacity': 0.15},
        tooltip=NAME,
    ).add_to(m)
    display(m)

boundary_gdf is not set — run one of the Method cells above first.
If using Method D, check that SEARCH_NAME matches a name in the printed table.
CPU times: user 192 μs, sys: 0 ns, total: 192 μs
Wall time: 187 μs


---
## Save

Saves the boundary to `boundaries/{name}.geojson` — the format expected by all
subsequent notebooks and by `pipeline.py`.

The file contains a single polygon in **WGS-84 (EPSG:4326)**.

In [ ]:
%%time
boundary_gdf.to_crs('EPSG:4326').to_file(OUTPUT_PATH, driver='GeoJSON')
print(f'Saved → {OUTPUT_PATH}')
print(f'  Features : {len(boundary_gdf)}')
print(f'  Bounds   : W={west:.5f}  S={south:.5f}  E={east:.5f}  N={north:.5f}')
print()
print('Next step: open notebook 1_filter_pbf.ipynb')
print(f'  Set NAME = "{NAME}" and BOUNDARY_PATH to this file.')